# EDA Weather Demand

Notebook para visualizar a relacao entre clima diario consolidado e demanda diaria de taxi.

A tabela usada aqui e a Gold `daily_weather_demand`, com grao de 1 linha por dia.

## Ideia

Para responder se clima afeta demanda, a unidade de analise e o dia:

```text
data + clima consolidado de NYC + qtd_corridas do dia
```

A `fact_trips` continua sendo melhor para Power BI dimensional. Esta tabela diaria e melhor para EDA e ML.

In [1]:
from v2.config.paths import DELTA_ROOT, daily_weather_demand_gold_dir

official_path = daily_weather_demand_gold_dir(2025)
dev_path = DELTA_ROOT / "dev" / "gold" / "daily_weather_demand" / "2025_01"

path = dev_path if (dev_path / "_delta_log").exists() else official_path
is_dev = path == dev_path

print(f"Official path: {official_path}")
print(f"Dev path     : {dev_path}")
print(f"Using path   : {path}")

if not (path / "_delta_log").exists():
    raise FileNotFoundError(f"Gold daily_weather_demand nao encontrada em {path}")

if is_dev:
    print("ATENCAO: usando amostra dev. Serve para validar a logica, nao para conclusao estatistica final.")

Official path: /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/gold/daily_weather_demand/2025
Dev path     : /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/gold/daily_weather_demand/2025_01
Using path   : /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/gold/daily_weather_demand/2025_01
ATENCAO: usando amostra dev. Serve para validar a logica, nao para conclusao estatistica final.


In [2]:
from v2.config.spark import create_spark

spark = create_spark("NotebookEDAWeatherDemand")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/31 15:06:00 WARN Utils: Your hostname, deskdev, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/31 15:06:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/delldev/projetos/nyc-taxi-lakehouse/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/delldev/.ivy2.5.2/cache
The jars for the packages stored in: /home/delldev/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-565cfbfc-9245-4688-a335-3ac79e0e2fd6;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in ce

In [ ]:
df = spark.read.format("delta").load(str(path))
df.printSchema()

## Cobertura

In [ ]:
from pyspark.sql import functions as F

df.select(
    F.count("*").alias("linhas"),
    F.countDistinct("data").alias("dias_distintos"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
    F.sum(F.col("sem_corridas").cast("int")).alias("dias_sem_corridas"),
    F.sum(F.col("sem_clima").cast("int")).alias("dias_sem_clima"),
    F.sum(F.col("registro_alinhamento_incompleto").cast("int")).alias("dias_alinhamento_incompleto"),
).show(truncate=False)

## Base de analise

Quando estiver usando a amostra dev, muitos dias ficam com `qtd_corridas = 0` porque a amostra nao contem o ano inteiro de corridas. Para nao distorcer a leitura, os proximos agrupamentos usam apenas dias com corrida na amostra.

Quando a Gold oficial completa existir, todos os dias entram na analise.

In [ ]:
analysis_df = df.filter(F.col("qtd_corridas") > 0) if is_dev else df

analysis_df.select(
    F.count("*").alias("dias_usados_na_analise"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
    F.round(F.avg("qtd_corridas"), 2).alias("media_corridas_dia"),
    F.min("qtd_corridas").alias("min_corridas_dia"),
    F.max("qtd_corridas").alias("max_corridas_dia"),
).show(truncate=False)

## Amostra diaria

In [ ]:
analysis_df.select(
    "data",
    "dia_semana_num",
    "fim_de_semana",
    "qtd_corridas",
    "qtd_estacoes",
    "precipitacao_media_mm",
    "precipitacao_max_mm",
    "temp_media_c",
    "neve_media_mm",
    "teve_chuva",
    "teve_neve",
    "categoria_chuva",
    "categoria_temperatura",
).orderBy("data").show(40, truncate=False)

## Demanda por chuva

In [ ]:
analysis_df.groupBy("teve_chuva", "categoria_chuva").agg(
    F.count("*").alias("dias"),
    F.round(F.avg("qtd_corridas"), 2).alias("media_corridas"),
    F.expr("percentile_approx(qtd_corridas, 0.5)").alias("mediana_corridas"),
    F.round(F.avg("precipitacao_media_mm"), 2).alias("precipitacao_media_mm"),
    F.round(F.max("precipitacao_max_mm"), 2).alias("precipitacao_max_mm"),
).orderBy("categoria_chuva").show(truncate=False)

## Demanda por temperatura

In [ ]:
analysis_df.groupBy("categoria_temperatura").agg(
    F.count("*").alias("dias"),
    F.round(F.avg("qtd_corridas"), 2).alias("media_corridas"),
    F.expr("percentile_approx(qtd_corridas, 0.5)").alias("mediana_corridas"),
    F.round(F.avg("temp_media_c"), 2).alias("temp_media_c"),
    F.round(F.min("temp_media_c"), 2).alias("temp_min_media_c"),
    F.round(F.max("temp_media_c"), 2).alias("temp_max_media_c"),
).orderBy("temp_media_c").show(truncate=False)

## Demanda por neve

In [ ]:
analysis_df.groupBy("teve_neve").agg(
    F.count("*").alias("dias"),
    F.round(F.avg("qtd_corridas"), 2).alias("media_corridas"),
    F.expr("percentile_approx(qtd_corridas, 0.5)").alias("mediana_corridas"),
    F.round(F.avg("neve_media_mm"), 2).alias("neve_media_mm"),
    F.round(F.max("neve_max_mm"), 2).alias("neve_max_mm"),
).orderBy("teve_neve").show(truncate=False)

## Correlacoes simples

Correlacao nao prova causalidade. Aqui ela serve como primeiro sinal para entender quais variaveis climaticas parecem se mover junto com a demanda.

In [ ]:
weather_features = [
    "precipitacao_media_mm",
    "precipitacao_max_mm",
    "temp_media_c",
    "temp_max_media_c",
    "temp_min_media_c",
    "neve_media_mm",
    "neve_max_mm",
    "qtd_estacoes",
]

rows = []
for feature in weather_features:
    value = analysis_df.stat.corr("qtd_corridas", feature)
    rows.append((feature, None if value is None else round(value, 4)))

spark.createDataFrame(rows, ["variavel_clima", "correlacao_com_qtd_corridas"]).show(
    truncate=False
)

## Grafico opcional

Se `pandas` e `matplotlib` estiverem disponiveis no ambiente, este bloco desenha demanda e chuva por dia. Se nao estiverem, os agrupamentos Spark acima ja bastam para a primeira leitura.

In [ ]:
try:
    import matplotlib.pyplot as plt

    pdf = analysis_df.select(
        "data", "qtd_corridas", "precipitacao_media_mm", "temp_media_c"
    ).orderBy("data").toPandas()

    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(pdf["data"], pdf["qtd_corridas"], label="qtd_corridas", color="#2563eb")
    ax1.set_ylabel("qtd_corridas")
    ax1.tick_params(axis="x", rotation=45)

    ax2 = ax1.twinx()
    ax2.bar(
        pdf["data"],
        pdf["precipitacao_media_mm"],
        label="precipitacao_media_mm",
        color="#64748b",
        alpha=0.3,
    )
    ax2.set_ylabel("precipitacao_media_mm")

    fig.tight_layout()
    plt.show()
except ImportError as exc:
    print(f"Grafico pulado: dependencia nao instalada ({exc}).")

## Caminho para ML

Nesta tabela, o alvo mais direto e `qtd_corridas`.

Features candidatas:

```text
precipitacao_media_mm
precipitacao_max_mm
temp_media_c
neve_media_mm
teve_chuva
teve_neve
fim_de_semana
dia_semana_num
mes
```

O modelo deve ser treinado depois da Gold completa, preferencialmente no Databricks, para evitar conclusoes em cima da amostra dev.

In [ ]:
spark.stop()